# Volumes

This example builds a small detector as a hierarchy of volumes, as in Geant4: logical volumes
(a shape and a material) placed inside each other with a transformation. It then draws it.

## Loading the modules

In [ ]:
using Geom4hep, Geom4hep.Units
using CairoMakie: CairoMakie, Figure, LScene

const T = Float64;

## Materials
Only the density of the materials is used for now (in g/cm³, as in GDML files).

In [ ]:
vacuum = Material{T}("vacuum"; density=0)
air    = Material{T}("air"; density=0.0012)
iron   = Material{T}("iron"; density=7.87)
lead   = Material{T}("lead"; density=11.35)
water  = Material{T}("water"; density=1.0);

## Volumes
A `Volume` is a shape filled with a material: the logical volume of Geant4.

In [ ]:
world  = Volume{T}("world", Box{T}(1m, 1m, 1m), air)
plate  = Volume{T}("plate", Box{T}(10cm, 20cm, 30cm), iron)
brick  = Volume{T}("brick", Box{T}(4cm, 4cm, 4cm), lead)
tank   = Volume{T}("tank", Trd{T}(25cm, 10cm, 25cm, 10cm, 10cm), water)
ball   = Volume{T}("ball", Sphere{T}(0, 8cm), lead)
pipe   = Volume{T}("pipe", Tube{T}(4cm, 5cm, 80cm, 0, 2π), iron);

## Placements
`place_daughter!(mother, transformation, daughter)` places a daughter volume in a mother volume.
`Transformation3D{T}(x, y, z)` places it at (x, y, z) of the mother frame, and the optional
rotation rotates the frame of the mother into the frame of the daughter. The same volume can be
placed several times: here the plate (with a lead brick inside) is placed at the 8 corners of a
cube, with different rotations.

In [ ]:
place_daughter!(plate, Transformation3D{T}(0, 0, 0), brick)
for (i, (x, y, z)) in enumerate(Iterators.product((-50cm, 50cm), (-50cm, 50cm), (-50cm, 50cm)))
    rotation = RotXYZ{T}(i % 3 == 0 ? π/4 : 0, i % 3 == 1 ? π/4 : 0, i % 3 == 2 ? π/4 : 0)
    place_daughter!(world, Transformation3D{T}(x, y, z, rotation), plate)
end
place_daughter!(tank, Transformation3D{T}(0, 0, 0), ball)
place_daughter!(world, Transformation3D{T}(0, 0, 0), tank)
place_daughter!(world, Transformation3D{T}(0, 0, 40cm, π/2, 0, 0), pipe) # along y, above the tank

Printing a volume shows its shape, material and daughters

In [ ]:
world

The daughters of a volume are `PlacedVolume`s, with the volume and its transformation.
Indexing a volume goes down the hierarchy: `world[1]` is the volume of the first daughter of
the world, and `world[1, 1]` the volume of its first daughter.

In [ ]:
world[1, 1]

## Assemblies
An `Assembly` groups volumes without a shape of its own: placing it places each of its volumes
in the mother. Here two small iron bars are placed below the tank.

In [ ]:
bar = Volume{T}("bar", Box{T}(20cm, 2cm, 2cm), iron)
group = Assembly{T}("bars")
place_daughter!(group, Transformation3D{T}(0, -5cm, 0), bar)
place_daughter!(group, Transformation3D{T}(0, 5cm, 0), bar)
place_daughter!(world, Transformation3D{T}(0, 0, -30cm), group)
length(world.daughters)

## Drawing
`draw!` draws the volumes of the hierarchy with grey levels from light (outer volumes) to dark
(deep daughters). The world volume itself is not drawn.

In [ ]:
fig = Figure(size=(800, 600))
draw!(LScene(fig[1, 1], show_axis=false), world)
fig

With `wireframe=true` the edges of the meshes are drawn instead, and `maxlevel` limits the depth.

In [ ]:
fig = Figure(size=(800, 600))
draw!(LScene(fig[1, 1], show_axis=false), world; wireframe=true, maxlevel=2)
fig

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*